# ──  Dry/wet edges


# STEP 1: Load your long-format data ──────────────────────────────────────

Required columns: id, year, reach_numb, forest_type, ndvi_*, lst_*
 If you only have vhi columns, you need to go back to GEE and export 
 raw NDVI and LST before VHI was computed

 For this workflow, we need RAW NDVI and LST per pixel per year

If not, the sub-regional correction can be applied to VCI/TCI components

# STEP 2: Define sub-regions ──────────────────────────────────────────────
 Option A: by reach (most ecologically meaningful for your study)
 Option B: by forest type
 Option C: by distance quartile to main channel
 We use reach as primary sub-region

# STEP 3: Compute sub-regional dry and wet edges ──────────────────────────
The dry edge = upper envelope of LST for a given NDVI bin (high LST, low NDVI)
 The wet edge = lower envelope of LST for a given NDVI bin (low LST, high NDVI)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:

subregion_col = 'reach_numb'  


def fit_edges(df_sub, ndvi_col='ndvi', lst_col='lst', n_bins=20):
    """
    Fit dry and wet edges of the LST-NDVI feature space for a sub-region.
    Returns dry_slope, dry_intercept, wet_slope, wet_intercept.
    """
    df_sub = df_sub.dropna(subset=[ndvi_col, lst_col]).copy()
    
    # Bin NDVI into equal-width bins
    df_sub['ndvi_bin'] = pd.cut(df_sub[ndvi_col], bins=n_bins)
    
    dry_points = []  # max LST per NDVI bin = dry edge
    wet_points = []  # min LST per NDVI bin = wet edge
    
    for bin_label, group in df_sub.groupby('ndvi_bin', observed=True):
        if len(group) < 5:
            continue
        mid_ndvi = bin_label.mid
        dry_points.append((mid_ndvi, group[lst_col].quantile(0.95)))  # 95th = dry
        wet_points.append((mid_ndvi, group[lst_col].quantile(0.05)))  # 5th = wet
    
    dry_df = pd.DataFrame(dry_points, columns=['ndvi', 'lst']).dropna()
    wet_df = pd.DataFrame(wet_points, columns=['ndvi', 'lst']).dropna()
    
    # Fit linear regression to each edge
    dry_slope, dry_int, _, _, _ = stats.linregress(dry_df['ndvi'], dry_df['lst'])
    wet_slope, wet_int, _, _, _ = stats.linregress(wet_df['ndvi'], wet_df['lst'])
    
    return {
        'dry_slope': dry_slope, 'dry_intercept': dry_int,
        'wet_slope': wet_slope, 'wet_intercept': wet_int
    }


# ── STEP 4: Apply sub-regional edges to compute corrected VHI ───────────────
def compute_regional_vhi(df, ndvi_col='ndvi', lst_col='lst', 
                          subregion_col='reach_numb', n_bins=20):
    """
    Compute VHI using sub-regional dry/wet edge fitting.
    Returns df with new columns: vci_regional, tci_regional, vhi_regional
    """
    df = df.copy()
    df['vci_regional'] = np.nan
    df['tci_regional'] = np.nan
    
    for region, group_idx in df.groupby(subregion_col).groups.items():
        group = df.loc[group_idx]
        
        # --- VCI: normalize NDVI within sub-region ---
        ndvi_min = group[ndvi_col].quantile(0.05)  # 5th percentile = historical low
        ndvi_max = group[ndvi_col].quantile(0.95)  # 95th percentile = historical high
        
        vci = (group[ndvi_col] - ndvi_min) / (ndvi_max - ndvi_min)
        vci = vci.clip(0, 1)
        df.loc[group_idx, 'vci_regional'] = vci
        
        # --- TCI: fit dry/wet edges within sub-region ---
        edges = fit_edges(group, ndvi_col=ndvi_col, lst_col=lst_col, n_bins=n_bins)
        
        lst_dry = (edges['dry_slope'] * group[ndvi_col] + 
                   edges['dry_intercept'])  # predicted dry-edge LST at this NDVI
        lst_wet = (edges['wet_slope'] * group[ndvi_col] + 
                   edges['wet_intercept'])  # predicted wet-edge LST at this NDVI
        
        tci = (lst_dry - group[lst_col]) / (lst_dry - lst_wet)
        tci = tci.clip(0, 1)
        df.loc[group_idx, 'tci_regional'] = tci
        
    df['vhi_regional'] = 0.5 * df['vci_regional'] + 0.5 * df['tci_regional']
    
    return df





In [ ]:

# ── STEP 5: Run it ──────────────────────────────────────────────────────────
# You need raw NDVI and LST columns in df_long
# If your current df_long only has vhi columns, add this note:
# Go back to GEE and export: NDVI, LST per pixel per year alongside VHI
# Then merge back to df_long on ['id', 'year']

# Assuming you have them:
df_corrected = compute_regional_vhi(
    df_long, 
    ndvi_col='ndvi',       # your raw NDVI column name
    lst_col='lst',         # your raw LST column name
    subregion_col='reach_numb',
    n_bins=20
)

print(df_corrected[['id', 'year', 'reach_numb', 
                     'vhi', 'vhi_regional']].head(20))



In [ ]:
# ── STEP 6: Compare original vs regional VHI ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, 
                           ['vhi', 'vhi_regional'],
                           ['Original VHI (global normalization)', 
                            'Regional VHI (sub-regional edge fitting)']):
    df_corrected.groupby(['year', 'reach_numb'])[col].mean().unstack().plot(ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Mean VHI')
    ax.legend(title='Reach')
    ax.axhline(0.5, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('vhi_comparison_regional.png', dpi=150)
plt.show()

# ── STEP 7: Diagnostic — plot LST-NDVI feature space with edges ─────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (reach, group) in zip(axes, df_long.groupby('reach_numb')):
    group_clean = group.dropna(subset=['ndvi', 'lst'])
    
    ax.scatter(group_clean['ndvi'], group_clean['lst'], 
               alpha=0.02, s=1, color='steelblue')
    
    # Fit and plot edges
    edges = fit_edges(group_clean, n_bins=20)
    ndvi_range = np.linspace(group_clean['ndvi'].min(), 
                              group_clean['ndvi'].max(), 100)
    
    ax.plot(ndvi_range, 
            edges['dry_slope'] * ndvi_range + edges['dry_intercept'],
            'r-', label='Dry edge', linewidth=2)
    ax.plot(ndvi_range,
            edges['wet_slope'] * ndvi_range + edges['wet_intercept'],
            'b-', label='Wet edge', linewidth=2)
    
    ax.set_title(f'Reach {reach}')
    ax.set_xlabel('NDVI')
    ax.set_ylabel('LST (°C or K)')
    ax.legend(fontsize=8)

plt.suptitle('LST-NDVI Feature Space with Sub-Regional Dry/Wet Edges', y=1.02)
plt.tight_layout()
plt.savefig('lst_ndvi_edges_by_reach.png', dpi=150)
plt.show()